# Crossmatch Ariel Targets vs MuSCAT-db Targets

This notebook performs a spatial crossmatch between exoplanet targets from the hypothetical ARIEL mission candidate list (`ariel_targets.csv`) and observed targets in the local MuSCAT-db database (`muscatdb_targets.csv`). The goal is to identify which ARIEL targets have already been observed by MuSCAT instruments, leveraging `astropy.coordinates` for robust celestial coordinate handling.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord, Angle
import astropy.units as u
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = Path("../data")
SEPARATION_THRESHOLD = 2.0  # arcsec

## 1. Load Data

We load the ARIEL candidate targets and the MuSCAT-db observed targets from CSV files. Note: `muscatdb_targets.csv` is assumed to be an export from the `muscat.db` `targets` table. The consistency of this export with the live database can be verified using the `notebooks/_csv_audit.py` script.

In [ ]:
try:
    ariel_df = pd.read_csv(DATA_DIR / "ariel_targets.csv")
    # The muscatdb_targets.csv might have inconsistent formatting, so using on_bad_lines="skip"
    muscat_df = pd.read_csv(DATA_DIR / "muscatdb_targets.csv", on_bad_lines="skip")
except FileNotFoundError:
    print("Error: Ensure 'ariel_targets.csv' and 'muscatdb_targets.csv' are in the ../data/ directory.")
    # Create dummy dataframes for demonstration if files are not found
    print("Generating dummy data for demonstration.")
    ariel_data = {
        'Star Name': ['ARIEL-1', 'ARIEL-2', 'TOI-2887', 'ARIEL-4', 'ARIEL-5'],
        'Star RA': [10.0, 20.0, 121.985508, 40.0, 50.0],
        'Star Dec': [1.0, 2.0, -39.014522, 4.0, 5.0],
        'TIC ID': [123, 456, 134183038, 789, 101],
    }
    ariel_df = pd.DataFrame(ariel_data)
    muscat_data = {
        'object': ['MUSCAT-A', 'MUSCAT-B', 'TOI2887', 'MUSCAT-D', 'MUSCAT-E'],
        'ra': ['00:40:00.00', '01:20:00.00', '08:07:56.526', '02:40:00.00', '03:20:00.00'],
        'dec': ['+01:00:00.0', '+02:00:00.0', '-39:00:52.33', '+04:00:00.0', '+05:00:00.0'],
        'n_frames': [100, 200, 300, 400, 500],
    }
    muscat_df = pd.DataFrame(muscat_data)

print(f"Loaded {len(ariel_df)} ARIEL targets and {len(muscat_df)} MuSCAT-db targets.")
print("\nARIEL targets (head):")
print(ariel_df.head())
print("\nMuSCAT-db targets (head):")
print(muscat_df.head())


## 2. Clean and Standardize Coordinates

MuSCAT-db's RA/Dec are often in sexagesimal format. We convert these to decimal degrees and drop any rows with malformed or missing coordinates. ARIEL coordinates are assumed to be in decimal degrees.

In [ ]:
def convert_sexagesimal_to_degrees(ra_series, dec_series):
    ra_deg = []
    dec_deg = []
    for ra_str, dec_str in zip(ra_series, dec_series):
        try:
            # Angle constructor can handle various formats, including sexagesimal strings
            ra_val = Angle(ra_str, unit=u.hourangle).deg
            dec_val = Angle(dec_str, unit=u.deg).deg
            ra_deg.append(ra_val)
            dec_deg.append(dec_val)
        except Exception:
            ra_deg.append(np.nan)
            dec_deg.append(np.nan)
    return np.array(ra_deg, dtype=float), np.array(dec_deg, dtype=float)

ariel_clean = ariel_df.dropna(subset=["Star RA", "Star Dec"]).reset_index(drop=True).copy()
muscat_clean = muscat_df.dropna(subset=["ra", "dec"]).copy()

# Convert MuSCAT-db sexagesimal coordinates to decimal degrees
muscat_clean["ra_deg"], muscat_clean["dec_deg"] = convert_sexagesimal_to_degrees(
    muscat_clean["ra"].astype(str), muscat_clean["dec"].astype(str)
)

muscat_clean = muscat_clean.dropna(subset=["ra_deg", "dec_deg"]).reset_index(drop=True)

print(f"Cleaned ARIEL targets: {len(ariel_clean)}")
print(f"Cleaned MuSCAT-db targets: {len(muscat_clean)}")

## 3. Build `SkyCoord` Objects and Perform Crossmatch

We create `astropy.coordinates.SkyCoord` objects for both datasets and use `search_around_sky` to find matches within a specified angular separation threshold.

In [ ]:
coord_ariel = SkyCoord(
    ra=ariel_clean["Star RA"].values * u.deg,
    dec=ariel_clean["Star Dec"].values * u.deg,
)
coord_muscat = SkyCoord(
    ra=muscat_clean["ra_deg"].values * u.deg,
    dec=muscat_clean["dec_deg"].values * u.deg,
)

# search_around_sky returns indices as (argument-coords, self-coords),
# so calling on coord_ariel with coord_muscat yields (muscat_idx, ariel_idx).
idx_muscat, idx_ariel, d2d, _ = coord_ariel.search_around_sky(
    coord_muscat, SEPARATION_THRESHOLD * u.arcsec
)
n_matches = len(idx_ariel)
n_ariel_total = len(ariel_clean)
n_muscat_total = len(muscat_clean)
n_matched_ariel = len(set(idx_ariel))
n_matched_muscat = len(set(idx_muscat))

print(f"Separation threshold: {SEPARATION_THRESHOLD} arcsec")
print(f"Total ARIEL targets (with valid coords): {n_ariel_total}")
print(f"Total MuSCAT-db targets (with valid coords): {n_muscat_total}")
print(f"ARIEL targets with a match in MuSCAT-db: {n_matched_ariel}")
print(f"MuSCAT-db targets with a match in ARIEL: {n_matched_muscat}")
print(f"Total unique matched pairs: {n_matches}")

## 4. Inspect Matched Pairs

A DataFrame is constructed to show details of the matched targets from both catalogs, including their names, IDs, coordinates, and the separation.

In [ ]:
matched_ariel_names = ariel_clean.iloc[idx_ariel]["Star Name"].values
matched_muscat_names = muscat_clean.iloc[idx_muscat]["object"].values

match_df = pd.DataFrame({
    "ariel_idx": idx_ariel,
    "muscat_idx": idx_muscat,
    "sep_arcsec": d2d.arcsec.value,
    "ariel_name": matched_ariel_names,
    "ariel_tic": ariel_clean.iloc[idx_ariel]["TIC ID"].values,
    "ariel_ra_deg": ariel_clean.iloc[idx_ariel]["Star RA"].values,
    "ariel_dec_deg": ariel_clean.iloc[idx_ariel]["Star Dec"].values,
    "muscat_name": matched_muscat_names,
    "muscat_ra_deg": muscat_clean.iloc[idx_muscat]["ra_deg"].values,
    "muscat_dec_deg": muscat_clean.iloc[idx_muscat]["dec_deg"].values,
})
match_df = match_df.sort_values("sep_arcsec").reset_index(drop=True)

print("Top 10 matched pairs:")
print(match_df.head(10))

print("\nUnique MuSCAT-db targets that have an ARIEL match:")
print(match_df.muscat_name.unique())

## 5. Visualize Crossmatch Results

A sky plot to visualize the distribution of ARIEL and MuSCAT-db targets, highlighting the successfully crossmatched objects.

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(coord_ariel.ra.deg, coord_ariel.dec.deg, s=10, alpha=0.5, label='ARIEL Targets (All)', color='blue')
plt.scatter(coord_muscat.ra.deg, coord_muscat.dec.deg, s=10, alpha=0.5, label='MuSCAT-db Targets (All)', color='green')

# Plot matched targets
plt.scatter(coord_ariel[idx_ariel].ra.deg, coord_ariel[idx_ariel].dec.deg, 
            s=50, marker='o', edgecolors='red', facecolors='none', linewidth=1.5, label='Matched Targets')

plt.xlabel('Right Ascension (deg)')
plt.ylabel('Declination (deg)')
plt.title(f'ARIEL vs MuSCAT-db Target Crossmatch (Threshold: {SEPARATION_THRESHOLD}\')')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 6. Unmatched Targets

Identify targets that did not find a match in the other catalog. These represent unique observation opportunities or targets observed exclusively by one facility.

In [ ]:
matched_ariel_indices = set(idx_ariel)
matched_muscat_indices = set(idx_muscat)

unmatched_ariel_df = ariel_clean[~ariel_clean.index.isin(matched_ariel_indices)]
unmatched_muscat_df = muscat_clean[~muscat_clean.index.isin(matched_muscat_indices)]

print(f"ARIEL targets NOT found in MuSCAT-db: {len(unmatched_ariel_df)}")
print(unmatched_ariel_df[["Star Name", "Star RA", "Star Dec"]].head())

print(f"\nMuSCAT-db targets NOT found in ARIEL: {len(unmatched_muscat_df)}")
print(unmatched_muscat_df[["object", "ra", "dec"]].head())

## 7. Export Results

The matched pairs are exported to a CSV file for further analysis or record-keeping.

In [ ]:
output_path = DATA_DIR / "crossmatch_results.csv"
match_df.to_csv(output_path, index=False)
print(f"Matched pairs saved to {output_path}")

## 8. Conclusion

This notebook successfully crossmatched ARIEL candidate targets with existing MuSCAT-db observations based on celestial coordinates. The results provide a clear overview of common targets and highlight unique observation opportunities for both programs. This methodology can be adapted for crossmatching with other astronomical catalogs or observation plans.